# K-Means

## Introdução

O K-Means particiona $N$ observações não rotuladas em $K$ clusters. O algoritmo alterna duas etapas: atribuir cada ponto ao centróide mais próximo e recalcular cada centróide como a média dos seus pontos. Repetindo isso, a variância intra-cluster diminui até as atribuições se estabilizarem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Estilo para os gráficos
plt.style.use('seaborn-v0_8-whitegrid')

## Fundamentação Matemática

O K-Means minimiza a variância intra-cluster, também chamada de inércia ou WCSS (*Within-Cluster Sum of Squares*):

$$ J = \sum_{k=1}^{K} \sum_{i \in C_k} || \mathbf{x}_i - \boldsymbol{\mu}_k ||^2 $$

- $K$: número de clusters
- $C_k$: observações pertencentes ao cluster $k$
- $\mathbf{x}_i$: $i$-ésima observação
- $\boldsymbol{\mu}_k$: centróide (média) do cluster $k$
- $|| \mathbf{x}_i - \boldsymbol{\mu}_k ||^2$: distância Euclidiana ao quadrado entre os dois

### O Processo Iterativo

A otimização segue o esquema Expectation-Maximization (EM):

1. **Atribuição (E)**: cada observação vai para o cluster do centróide mais próximo. Esta etapa define as fronteiras de decisão.
    $$ C_k = \{ i : ||\mathbf{x}_i - \boldsymbol{\mu}_k||^2 \le ||\mathbf{x}_i - \boldsymbol{\mu}_j||^2 \quad \forall j, 1 \le j \le K \} $$
2. **Atualização (M)**: cada centróide passa a ser a média dos pontos atribuídos a ele.
    $$ \boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{i \in C_k} \mathbf{x}_i $$

Os dois passos se repetem até que as atribuições não mudem mais.

## Preparação dos Dados

Usaremos o dataset *Iris*, com medições de três espécies de flores. Ele é clássico em classificação, mas serve bem à clusterização: os rótulos reais nos permitem avaliar visualmente o resultado. Vamos usar duas características — comprimento e largura da pétala — para visualizar em 2D.

### Escolhendo as Features

O **pair plot** mostra um scatter plot para cada par de features, revelando a separação dos grupos nos diferentes subespaços 2D. Para o K-Means, procuramos pares com aglomerados distintos e compactos.

Aqui colorimos os pontos pelas espécies reais para validar a escolha — num problema não supervisionado real, esses rótulos não existiriam.

In [ ]:
import seaborn as sns
import pandas as pd

# Carregar o dataset Iris
iris = load_iris()
X = iris.data
y_true = iris.target

# Para facilitar o uso com o Seaborn, vamos criar um DataFrame do Pandas
iris_df = pd.DataFrame(X, columns=iris.feature_names)
iris_df['species'] = pd.Series(iris.target).map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

# Gerar o pairplot
# O hue='species' colore os pontos de acordo com a espécie real da flor
sns.pairplot(iris_df, hue='species', palette='viridis', diag_kind='kde')
plt.suptitle('Pair Plot do Dataset Iris', y=1.02) # Ajusta o título para não sobrepor
plt.show()

In [ ]:
# Carregar o dataset Iris
iris = load_iris()
X = iris.data
y_true = iris.target

# Para fins de visualização, vamos usar apenas as duas últimas features:
# Comprimento da pétala (petal length) e Largura da pétala (petal width)
X = X[:, 2:]

# Visualizar os dados
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c='gray', marker='o', s=50, label='Dados Originais')
plt.xlabel(iris.feature_names[2])
plt.ylabel(iris.feature_names[3])
plt.title('Dataset Iris - Características das Pétalas')
plt.legend()
plt.show()

## Implementação do K-Means

Vamos construir a classe `KMeans` do zero. Ela armazena os hiperparâmetros, os centróides e os rótulos, e implementa a inicialização dos centróides, a atribuição dos clusters, a atualização e o `fit`.

In [ ]:
class KMeans:
    def __init__(self, n_clusters=3, max_iter=100, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.centroids = None
        self.labels = None

    def _initialize_centroids(self, X):
        """
        Inicializa os centróides selecionando K pontos aleatórios do dataset.
        """
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        random_indices = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[random_indices]

    def _assign_clusters(self, X):
        """
        Atribui cada ponto de dado ao centróide mais próximo.
        """
        n_samples = X.shape[0]
        distances = np.zeros((n_samples, self.n_clusters))

        for i, centroid in enumerate(self.centroids):
            distances[:, i] = np.sum((X - centroid)**2, axis=1)
        
        self.labels = np.argmin(distances, axis=1)

    def _update_centroids(self, X):
        """
        Atualiza a posição de cada centróide com base na média dos pontos atribuídos a ele.
        """
        new_centroids = np.zeros((self.n_clusters, X.shape[1]))
        
        for i in range(self.n_clusters):
            cluster_points = X[self.labels == i]
            if len(cluster_points) > 0:
                new_centroids[i] = np.mean(cluster_points, axis=0)
            else:
                new_centroids[i] = self.centroids[i]
        
        self.centroids = new_centroids

    def fit(self, X):
        """
        Executa o algoritmo K-Means.
        """
        self._initialize_centroids(X)

        for _ in range(self.max_iter):
            old_centroids = self.centroids.copy()
            self._assign_clusters(X)
            self._update_centroids(X)
            if np.allclose(old_centroids, self.centroids):
                break

    def predict(self, X):
        """
        Atribui clusters para novos dados com base nos centróides aprendidos.
        """
        distances = np.zeros((X.shape[0], self.n_clusters))
        for i, centroid in enumerate(self.centroids):
            distances[:, i] = np.sum((X - centroid)**2, axis=1)
        
        return np.argmin(distances, axis=1)

## Treinando o Modelo

Instanciamos a classe com $K=3$ (o Iris tem 3 espécies) e chamamos o `fit`.

In [ ]:
# Instanciar e treinar o modelo K-Means
kmeans = KMeans(n_clusters=3, max_iter=150, random_state=42)
kmeans.fit(X)

### Extraindo os Resultados
Após o treinamento, os centróides finais e os rótulos ficam armazenados no objeto `kmeans`.

In [ ]:
# Obter os centróides finais e os rótulos dos clusters
final_centroids = kmeans.centroids
predicted_labels = kmeans.labels

print("Coordenadas dos Centróides Finais:")
print(final_centroids)

### Visualização dos Clusters
Cada cluster recebe uma cor; os centróides finais aparecem com um marcador destacado.

In [ ]:
# Configurar a figura para plots lado a lado
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Clusters encontrados pelo K-Means
axes[0].scatter(X[:, 0], X[:, 1], c=predicted_labels, s=50, cmap='viridis', edgecolor='k')
axes[0].scatter(final_centroids[:, 0], final_centroids[:, 1], s=250, marker='*', c='red', edgecolor='black', label='Centróides')
axes[0].set_title('Clusters Encontrados pelo K-Means')
axes[0].set_xlabel('Comprimento da Pétala (cm)')
axes[0].set_ylabel('Largura da Pétala (cm)')
axes[0].legend()
axes[0].grid(True)

# Plot 2: Classes Reais do Dataset
scatter = axes[1].scatter(X[:, 0], X[:, 1], c=y_true, s=50, cmap='viridis', edgecolor='k')
axes[1].set_title('Classes Reais (Ground Truth)')
axes[1].set_xlabel('Comprimento da Pétala (cm)')
axes[1].set_ylabel('Largura da Pétala (cm)')
axes[1].legend(handles=scatter.legend_elements()[0], labels=iris.target_names.tolist())
axes[1].grid(True)

plt.suptitle('Comparação: K-Means vs. Rótulos Reais', fontsize=16)
plt.show()

### Contagem de Acertos

Os rótulos dos clusters (0, 1, 2...) são arbitrários e não correspondem aos rótulos reais, então não dá para compará-los diretamente. Uma medida intuitiva é a **pureza**: assumimos que cada cluster representa a espécie mais frequente dentro dele e contamos quantos pontos realmente pertencem a ela.

In [ ]:
from scipy.stats import mode
import numpy as np

correct_predictions = 0
n_samples = X.shape[0]

# Para cada label de cluster (0, 1, 2...)
for i in range(kmeans.n_clusters):
    # 1. Encontra todos os pontos de dados que foram atribuídos ao cluster 'i'
    mask = (predicted_labels == i)
    
    # 2. Dentre esses pontos, descobre qual é a classe real mais comum (a moda)
    # Se a maioria dos pontos no cluster 'i' for da classe real 2, assumimos que 'i' corresponde a 2
    dominant_label = mode(y_true[mask], keepdims=True)[0][0]
    
    # 3. Conta quantos pontos nesse cluster realmente pertencem a essa classe dominante
    hits = np.sum(y_true[mask] == dominant_label)
    
    # 4. Adiciona essa contagem ao total de acertos
    correct_predictions += hits

print(f"Número de acertos: {correct_predictions} de {n_samples} pontos.")
print(f"Taxa de acerto: {(correct_predictions / n_samples):.2%}")

### Regiões de Decisão

Os centróides particionam o espaço de características em regiões — um Diagrama de Voronoi. Para visualizá-las, criamos uma malha (`meshgrid`) cobrindo o gráfico, prevemos o cluster de cada ponto da malha e colorimos o fundo.

In [ ]:
# Criar uma malha para cobrir o espaço do gráfico
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

# Usar o modelo para prever o cluster de cada ponto na malha
Z = kmeans.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plotar o resultado
plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
plt.scatter(X[:, 0], X[:, 1], c=predicted_labels, s=50, edgecolor='k')
plt.scatter(final_centroids[:, 0], final_centroids[:, 1], s=250, marker='*', c='red', edgecolor='black', label='Centróides')
plt.title('Regiões de Decisão do K-Means')
plt.xlabel('Comprimento da Pétala (cm)')
plt.ylabel('Largura da Pétala (cm)')
plt.legend()
plt.show()

## O Método do Cotovelo (Elbow Method)

Fixamos $K=3$ porque já conhecíamos o Iris; num problema real, $K$ é desconhecido. A heurística do cotovelo roda o K-Means para um intervalo de valores de $K$ e registra a inércia de cada um:

$$ \text{Inércia} = \sum_{k=1}^{K} \sum_{i \in C_k} || \mathbf{x}_i - \boldsymbol{\mu}_k ||^2 $$

A inércia sempre diminui conforme $K$ cresce. O "cotovelo" — o ponto a partir do qual a queda passa a ser marginal — é um bom candidato a $K$ ótimo.

In [ ]:
# Implementando o Método do Cotovelo com visualização iterativa
k_range = range(1, 11)
inertias = []
clustering_results = []

for k in k_range:
    model = KMeans(n_clusters=k, max_iter=150, random_state=42)
    model.fit(X)
    
    # Calcular a inércia (WCSS)
    current_inertia = 0
    for i in range(k):
        # Seleciona os pontos pertencentes ao cluster i
        cluster_points = X[model.labels == i]
        # Calcula a soma das distâncias quadradas ao centróide do cluster i
        current_inertia += np.sum((cluster_points - model.centroids[i])**2)
    
    inertias.append(current_inertia)
    clustering_results.append({'labels': model.labels, 'centroids': model.centroids})

# Plotar o gráfico do cotovelo
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, 'bo-')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (WCSS)')
plt.title('Método do Cotovelo para Encontrar o K Ótimo')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# Plotar os resultados da clusterização para cada K
fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.ravel()

for i, k in enumerate(k_range):
    result = clustering_results[i]
    labels = result['labels']
    centroids = result['centroids']
    
    axes[i].scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis', edgecolor='k')
    axes[i].scatter(centroids[:, 0], centroids[:, 1], marker='*', s=150, c='red', edgecolor='black')
    axes[i].set_title(f'K = {k}')
    axes[i].grid(False)

plt.suptitle('Visualização da Clusterização para Diferentes Valores de K', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Análise dos Gráficos

O cotovelo aparece nitidamente em $K=3$: a partir daí, a inércia cai pouco. Os scatter plots confirmam:

* $K=1$: um único grupo.
* $K=2$: separa setosa, mas mantém versicolor e virginica juntas.
* $K=3$: corresponde bem à estrutura real dos dados.
* $K>3$: subdivide clusters já coesos, um "overfitting" da clusterização.

## Aplicação 1: Compressão de Imagem (Quantização Vetorial)

Cada pixel de uma imagem colorida é um vetor RGB, ou seja, um ponto num espaço 3D — e uma imagem pode ter centenas de milhares de cores únicas. Aplicando o K-Means a esses pixels, os $K$ centróides formam uma nova **paleta de cores**, e cada pixel é substituído pela cor do seu centróide.

O resultado se parece muito com a imagem original, mas usa só $K$ cores. A compressão vem daí: em vez do RGB de cada pixel, basta guardar a paleta de $K$ cores e, por pixel, um índice de 0 a $K-1$.

### Carregando e Preparando a Imagem
Usamos uma imagem de exemplo do `scikit-learn` (`load_sample_image`), o que dispensa qualquer download. Em seguida remodelamos a matriz (altura × largura × 3 canais) para o formato 2D (nº de pixels × 3) esperado pela nossa implementação.

In [ ]:
from sklearn.datasets import load_sample_image

# O scikit-learn traz duas imagens de exemplo: 'china.jpg' e 'flower.jpg'
source_image = load_sample_image('china.jpg')

# Normalizar os valores dos pixels para o intervalo [0, 1]
source_image = source_image / 255.0

# Obter as dimensões da imagem
height, width, channels = source_image.shape

# Remodelar a imagem para ser uma lista de pixels (vetores RGB)
pixel_list = source_image.reshape(-1, channels)

print(f"Dimensões originais da imagem: {source_image.shape}")
print(f"Dimensões da lista de pixels: {pixel_list.shape}")
print(f"Número de cores únicas na imagem original: {len(np.unique(pixel_list, axis=0))}")

### Criando a Paleta de Cores
Aplicamos o K-Means aos pixels. Quanto mais clusters, mais detalhes preservados.

In [ ]:
# Definir o número de clusters (cores na nova paleta)
K_compression = 8

# Instanciar e treinar um modelo K-Means específico para a compressão
compression_kmeans = KMeans(n_clusters=K_compression, max_iter=100, random_state=42)
compression_kmeans.fit(pixel_list)

# Os centróides encontrados formam a nossa nova paleta de cores
color_palette_compression = compression_kmeans.centroids

### Reconstruindo a Imagem Comprimida
Cada pixel é substituído pela cor do centróide do seu cluster e a lista volta ao formato original da imagem.

In [ ]:
# Obter os rótulos de cluster para cada pixel
pixel_labels_compression = compression_kmeans.labels

# Criar a imagem comprimida substituindo cada pixel pelo seu centróide
compressed_pixel_list = color_palette_compression[pixel_labels_compression]

# Remodelar a lista de pixels de volta para o formato de imagem original
compressed_image = compressed_pixel_list.reshape(height, width, channels)

# Visualizar a imagem original e a imagem comprimida lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(source_image)
axes[0].set_title(f'Imagem Original ({len(np.unique(pixel_list, axis=0))} cores)')
axes[0].axis('off')

axes[1].imshow(compressed_image)
axes[1].set_title(f'Imagem Comprimida ({K_compression} cores)')
axes[1].axis('off')

plt.show()

## Aplicação 2: Segmentação de Imagem

Segmentar é particionar a imagem em regiões, simplificando sua representação para análise. Como o K-Means agrupa pixels de cores semelhantes, os clusters correspondem naturalmente a regiões com perfis de cor distintos. Aqui poucos clusters funcionam melhor, já que queremos as macro-regiões.

### Carregando a Imagem para Segmentação
Usamos a segunda imagem de exemplo do `scikit-learn`, processada da mesma forma que a anterior.

In [ ]:
# Segunda imagem de exemplo do scikit-learn
segmentation_source_image = load_sample_image('flower.jpg')

# Normalizar os valores dos pixels para o intervalo [0, 1]
segmentation_source_image = segmentation_source_image / 255.0

# Obter as dimensões da imagem para segmentação
seg_height, seg_width, seg_channels = segmentation_source_image.shape

# Remodelar a imagem para ser uma lista de pixels (vetores RGB)
pixel_list_segmentation = segmentation_source_image.reshape(-1, seg_channels)

print(f"Dimensões originais da imagem para segmentação: {segmentation_source_image.shape}")
print(f"Dimensões da lista de pixels para segmentação: {pixel_list_segmentation.shape}")

### Treinando o Modelo de Segmentação
Agora com $K=3$, para identificar as principais regiões de cor.

In [ ]:
# Definir um número menor de clusters para a segmentação
K_segmentation = 3

# Instanciar e treinar um novo modelo K-Means específico para a segmentação
segmentation_kmeans = KMeans(n_clusters=K_segmentation, max_iter=100, random_state=42)
segmentation_kmeans.fit(pixel_list_segmentation)

### Visualizando o Mapa de Segmentação
Remodelando os `labels` de volta às dimensões da imagem, obtemos um mapa em que cada pixel guarda o ID do seu segmento.

In [ ]:
# Obter os rótulos e centróides do modelo de segmentação
pixel_labels_segmentation = segmentation_kmeans.labels
color_palette_segmentation = segmentation_kmeans.centroids

# Criar a imagem segmentada
segmentation_map = pixel_labels_segmentation.reshape(seg_height, seg_width)

# Visualizar a imagem original e a imagem segmentada
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(segmentation_source_image)
axes[0].set_title('Imagem Original')
axes[0].axis('off')

axes[1].imshow(segmentation_map, cmap='viridis')
axes[1].set_title(f'Imagem Segmentada em {K_segmentation} Regiões')
axes[1].axis('off')

plt.show()

## Exercícios

Use a classe `KMeans` que construímos no **Wine dataset**: análises químicas de vinhos de três cultivares diferentes, todos da mesma região da Itália. Será que o K-Means recupera os cultivares apenas a partir da química?

**Atenção:** o K-Means se baseia em distâncias, então é sensível à escala das variáveis. Padronize os dados com o `StandardScaler` do Scikit-Learn antes de treinar, para que features de maior magnitude não dominem o particionamento.

### Exercício 1: Análise e Seleção de Features

Carregue o dataset `wine`, use `seaborn.pairplot` para inspecionar as features e escolha o par que melhor separa os 3 grupos. Plote o scatter apenas desse par.

### Exercício 2: Encontrando o K Ótimo

Aplique o Método do Cotovelo às duas features escolhidas, plotando a inércia (WCSS) para K de 1 a 10. Qual parece ser o número ideal de clusters?

### Exercício 3: Clusterização e Avaliação

Treine o modelo com o K encontrado. Faça um gráfico com dois subplots — clusters encontrados vs. rótulos reais — e calcule a taxa de acertos, comentando o resultado.